# **Import of Libraries needed**

In [ ]:
import heapq
import time
import random
import statistics
import csv


# **Dijkstra and A* Search algorithms**

In [ ]:
# AI assistance: helped draft and explain this benchmark; review and test the code before submission.

def dijkstra(graph, start, destination, return_explored=False):
    # Check if the locations exist
    if start not in graph or destination not in graph:
        raise ValueError("Invalid starting or destination location")

    # Check that all edges have valid destinations and nonnegative costs
    for node in graph:
        for neighbor, cost in graph[node]:
            if neighbor not in graph:
                raise ValueError("Edge points to an unknown location")
            if cost < 0:
                raise ValueError("Negative route costs are not allowed")

    distances = {node: float("inf") for node in graph}
    previous = {node: None for node in graph}
    distances[start] = 0
    pq = [(0, start)]
    explored = 0

    while pq:
        current_dist, current_node = heapq.heappop(pq)

        if current_dist > distances[current_node]:
            continue

        explored += 1

        if current_node == destination:
            break

        for neighbor, weight in graph[current_node]:
            distance = current_dist + weight

            if distance < distances[neighbor]:
                distances[neighbor] = distance
                previous[neighbor] = current_node
                heapq.heappush(pq, (distance, neighbor))

    if distances[destination] == float("inf"):
        result = (None, float("inf"))
    else:
        route = []
        current = destination

        while current is not None:
            route.append(current)
            current = previous[current]

        route.reverse()
        result = (route, distances[destination])

    if return_explored:
        return result[0], result[1], explored

    return result


def astar(graph, start, destination, heuristic, return_explored=False):
    # Check if the locations exist
    if start not in graph or destination not in graph:
        raise ValueError("Invalid starting or destination location")

    # Check that all edges have valid destinations and nonnegative costs
    for node in graph:
        for neighbor, cost in graph[node]:
            if neighbor not in graph:
                raise ValueError("Edge points to an unknown location")
            if cost < 0:
                raise ValueError("Negative route costs are not allowed")

    distances = {node: float("inf") for node in graph}
    previous = {node: None for node in graph}
    distances[start] = 0

    # Priority is estimated total cost: actual cost so far + heuristic
    pq = [(heuristic(start, destination), 0, start)]
    explored = 0

    while pq:
        estimated_total, current_dist, current_node = heapq.heappop(pq)

        # Ignore an outdated queue entry
        if current_dist > distances[current_node]:
            continue

        explored += 1

        if current_node == destination:
            break

        for neighbor, weight in graph[current_node]:
            new_dist = current_dist + weight

            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                previous[neighbor] = current_node
                priority = new_dist + heuristic(neighbor, destination)
                heapq.heappush(pq, (priority, new_dist, neighbor))

    if distances[destination] == float("inf"):
        result = (None, float("inf"))
    else:
        route = []
        current = destination

        while current is not None:
            route.append(current)
            current = previous[current]

        route.reverse()
        result = (route, distances[destination])

    if return_explored:
        return result[0], result[1], explored

    return result


# **The Sample Game Map**

In [ ]:
game_map = {
    "Village": [
        ("Forest Path", 3),
        ("Mountain Path", 7)
    ],
    "Forest Path": [
        ("Village", 3),
        ("Cave", 4)
    ],
    "Mountain Path": [
        ("Village", 7),
        ("Cave", 2)
    ],
    "Cave": [
        ("Forest Path", 4),
        ("Mountain Path", 2)
    ]
}

route, cost = dijkstra(game_map, "Village", "Cave")
print("Dijkstra route:", route)
print("Total route cost:", cost)

# This small example uses a zero heuristic because it has no location-layout data.
astar_route, astar_cost = astar(
    game_map, "Village", "Cave", lambda node, goal: 0
)
print("A* route:", astar_route)
print("A* route cost:", astar_cost)


# **Automated tests for Dijkstra and A***

In [ ]:
# Test 1: Find the shortest route
route, cost = dijkstra(game_map, "Village", "Cave")
assert route == ["Village", "Forest Path", "Cave"]
assert cost == 7
print("Test 1 passed: Dijkstra shortest route")

# Test 2: Same starting point and destination
route, cost = dijkstra(game_map, "Village", "Village")
assert route == ["Village"]
assert cost == 0
print("Test 2 passed: Same start and destination")

# Test 3: Unreachable destination
disconnected_map = {"Village": [], "Cave": []}
route, cost = dijkstra(disconnected_map, "Village", "Cave")
assert route is None and cost == float("inf")
print("Test 3 passed: Unreachable destination")

# Test 4: Invalid destination
try:
    dijkstra(game_map, "Village", "Castle")
    assert False, "Expected a ValueError"
except ValueError:
    print("Test 4 passed: Invalid destination")

# Test 5: Negative route cost
negative_map = {"Village": [("Cave", -5)], "Cave": []}
try:
    dijkstra(negative_map, "Village", "Cave")
    assert False, "Expected a ValueError"
except ValueError:
    print("Test 5 passed: Negative route cost")

# Test 6: Empty graph
try:
    dijkstra({}, "Village", "Cave")
    assert False, "Expected a ValueError"
except ValueError:
    print("Test 6 passed: Empty graph rejected")

# Test 7: A* must find the same optimal cost as Dijkstra
astar_route, astar_cost = astar(
    game_map, "Village", "Cave", lambda node, goal: 0
)
assert astar_route[0] == "Village"
assert astar_route[-1] == "Cave"
assert astar_cost == 7
print("Test 7 passed: A* finds the same shortest cost")

print("\\nAll automated tests passed!")


# **Generate larger scene-location graphs for benchmarking**

In [ ]:
def generate_graph(number_of_nodes, extra_edges=2, seed=42):
    rng = random.Random(seed)

    graph = {f"Location_{i}": [] for i in range(number_of_nodes)}
    edge_costs = {}

    def add_edge(index_a, index_b, cost):
        # Keep one edge per pair and preserve the heuristic lower-bound rule.
        pair = tuple(sorted((index_a, index_b)))
        minimum_valid_cost = abs(index_a - index_b)
        cost = max(cost, minimum_valid_cost)

        if pair not in edge_costs or cost < edge_costs[pair]:
            edge_costs[pair] = cost

    # Connect each location to the next one so the graph is connected.
    for i in range(number_of_nodes - 1):
        add_edge(i, i + 1, rng.randint(1, 10))

    # Add extra connections to create alternative routes.
    for _ in range(extra_edges * number_of_nodes):
        a = rng.randrange(number_of_nodes)
        b = rng.randrange(number_of_nodes)

        if a != b:
            add_edge(a, b, rng.randint(1, 10))

    # Convert the edge list to an undirected adjacency list.
    for (a, b), cost in edge_costs.items():
        node_a = f"Location_{a}"
        node_b = f"Location_{b}"
        graph[node_a].append((node_b, cost))
        graph[node_b].append((node_a, cost))

    return graph


def location_heuristic(node, destination):
    # Lower bound for generated graphs: each edge costs at least the index gap.
    node_index = int(node.split("_")[1])
    destination_index = int(destination.split("_")[1])
    return abs(node_index - destination_index)


test_graph = generate_graph(10)
route, cost = dijkstra(test_graph, "Location_0", "Location_9")
print("Example route:", route)
print("Example cost:", cost)


# **Benchmark Dijkstra vs. A* Search**

In [ ]:
def benchmark_algorithms(sizes=(100, 500, 1000, 2000), repeats=5):
    results = []

    for size in sizes:
        # Both algorithms receive the exact same graph and endpoints.
        graph = generate_graph(size)
        start = "Location_0"
        destination = f"Location_{size - 1}"

        algorithms = [
            ("Dijkstra", lambda: dijkstra(
                graph, start, destination, return_explored=True
            )),
            ("A*", lambda: astar(
                graph, start, destination, location_heuristic,
                return_explored=True
            ))
        ]

        expected_cost = None

        for algorithm_name, run_algorithm in algorithms:
            # Warm-up and correctness check
            route, cost, explored = run_algorithm()

            assert route is not None
            assert route[0] == start
            assert route[-1] == destination

            # Confirm route cost matches the edges in the graph.
            route_cost = 0
            for i in range(len(route) - 1):
                current = route[i]
                next_location = route[i + 1]
                matching_costs = [
                    weight for neighbor, weight in graph[current]
                    if neighbor == next_location
                ]
                assert matching_costs, "Route contains an invalid edge"
                route_cost += min(matching_costs)

            assert route_cost == cost

            if expected_cost is None:
                expected_cost = cost
            else:
                assert cost == expected_cost, (
                    "Dijkstra and A* returned different route costs"
                )

            # Measure only algorithm runtime; graph creation is outside timing.
            times = []
            for _ in range(repeats):
                start_time = time.perf_counter()
                route, cost, explored = run_algorithm()
                end_time = time.perf_counter()
                times.append(end_time - start_time)

            results.append({
                "Algorithm": algorithm_name,
                "Locations": size,
                "Connections": sum(len(edges) for edges in graph.values()) // 2,
                "Median Runtime (ms)": round(
                    statistics.median(times) * 1000, 3
                ),
                "Route Cost": cost,
                "Nodes Explored": explored
            })

    return results


benchmark_results = benchmark_algorithms()

for result in benchmark_results:
    print(result)


# **Save both algorithms' benchmark results as CSV**

In [ ]:
with open("dijkstra_astar_benchmark_results.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=benchmark_results[0].keys())
    writer.writeheader()
    writer.writerows(benchmark_results)

print("Benchmark results saved to dijkstra_astar_benchmark_results.csv")


# **Download the benchmark CSV in Google Colab**

In [ ]:
from google.colab import files

files.download("dijkstra_astar_benchmark_results.csv")
